In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from import_ipynb import NotebookFinder  # type: ignore
import importlib
import joblib
import os
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report
from sklearn.metrics import make_scorer, recall_score

In [ ]:
# retrouver les dossiers
root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
hyperparameter_dir = os.path.join(root, "pneumonia_knn", "documents", "model", "hyperparameter")
utils_dir = os.path.join(root, "pneumonia_knn","notebooks", "utils")

# charger les fichiers
# --- pneumonia_knn\notebooks\visualisation\accuracy_metrics.ipynb
spec_metrics = NotebookFinder().find_spec("metrics", [utils_dir])
metrics = importlib.util.module_from_spec(spec_metrics)
spec_metrics.loader.exec_module(metrics)

# --- pneumonia_knn\notebooks\utils
spec_print = NotebookFinder().find_spec("prints", [utils_dir])
prints = importlib.util.module_from_spec(spec_print)
spec_print.loader.exec_module(prints)

In [ ]:
# 1. Créer le pipeline
pipeline = Pipeline([
    ('pca', PCA()),
    ('knn', KNeighborsClassifier())
])

# 2. Définir les hyperparamètres
param_grid = {
    'pca__n_components': [0.95],
    'knn__n_neighbors': [3, 5, 9],
    'knn__metric': ['euclidean', 'manhattan'],
    'knn__weights': ['distance'],
    'knn__algorithm': ['ball_tree']
}

In [ ]:
def load_grid(file_name, X_train, y_train):
    if os.path.exists(f'{hyperparameter_dir}/{file_name}'):
        # 2.5 Charger le GridSearchCV si il existe
        return joblib.load(f'{hyperparameter_dir}/{file_name}')
    else:
        # 2.5 Lancer GridSearchCV, sauvegarder le résultat et le retourner
        prints.bold("Process : Cross Validation - PCA + KNN")
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=2,
            scoring=metrics.get_scoring(),
            refit='recall',
            n_jobs=1,
            verbose=2
        )
        grid.fit(X_train, y_train)
        joblib.dump(grid, f'{hyperparameter_dir}/{file_name}')
        return grid

In [ ]:
def train_knn_pca(grid, X_test, y_test, X_test_2d_pca, X_test_2d_tsne):
    # 4. Récupérer le meilleur hyperparamètre
    knn_pca = grid.best_estimator_
    pca = knn_pca.named_steps['pca']

    # 5. Prédire et évaluer les résultats
    y_pred_pca = knn_pca.predict(X_test)
    metrics_results_cross_val = pd.DataFrame(grid.cv_results_)
    
    # 6. Afficher les résultats
    prints.bold_red("Résultats liés à l'hyperparamètres tuning (GridSearchCV) :")
    metrics.plot_grid_search_metric_overview_table(metrics_results_cross_val, 'Résultats GridSearchCV - KNN + PCA : métriques principales (GridSearchCV)')

    prints.bold_red("Résultats liés à la prédiction :")
    metrics.plot_knn_prediction_scatter(y_test, y_pred_pca, X_test_2d_pca, "PCA")
    metrics.plot_knn_prediction_scatter(y_test, y_pred_pca, X_test_2d_tsne, "PCA + TSNE")
    metrics.plot_prediction_metrics_bar(y_test, y_pred_pca, "PCA")
    metrics.plot_true_positives_by_class(y_test, y_pred_pca, title="KNN + PCA")
    metrics.plot_pca_variance_explained(pca)
    metrics.plot_pca_components(knn_pca.named_steps['pca'])
    print(classification_report(y_test, y_pred_pca, 
      target_names=['NORMAL', 'BACTERIA', 'VIRUS']))

In [ ]:
def implementation_with_PCA(file_name, X_train, X_test, y_train, y_test):
    # 3. Charger le GridSearchCV
    grid = load_grid(file_name, X_train, y_train)

    # Créer les nouvelles données transformées par PCA pour les utiliser dans les autres algorithmes
    X_train_pca = grid.best_estimator_.named_steps['pca'].transform(X_train)
    X_test_pca = grid.best_estimator_.named_steps['pca'].transform(X_test)

    X_test_2d_pca = X_test_pca[:, :2]
    tsne = TSNE(n_components=2, random_state=42)
    X_test_2d_tsne = tsne.fit_transform(X_test_pca)
    
    # 4. Entraîner le modèle KNN avec PCA
    train_knn_pca(grid, X_test, y_test, X_test_2d_pca, X_test_2d_tsne)

    
    return X_train_pca, X_test_pca